In [ ]:
# This R environment comes with many helpful analytics packages installed
# It is defined by the kaggle/rstats Docker image: https://github.com/kaggle/docker-rstats
# For example, here's a helpful package to load

library(tidyverse) # metapackage of all tidyverse packages

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

list.files(path = "../input")

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
install.packages(c("igraph", "ergm", "network"))

library(igraph)   # 网络创建与可视化
library(ergm)     # 指数随机图模型
library(network)  # ergm依赖的网络数据格式

set.seed(42)

In [ ]:
# 1. 随机网络 vs 真实世界网络

er_net <- erdos.renyi.game(100, p = 0.06)
ba_net <- simplify(sample_pa(100, power = 1, m = 3, directed = FALSE))

cat("聚集系数 ER:", round(transitivity(er_net), 3),
    " BA:", round(transitivity(ba_net), 3), "\n")
cat("平均路径 ER:", round(mean_distance(er_net), 3),
    " BA:", round(mean_distance(ba_net), 3), "\n")

par(mfrow = c(1, 2))
hist(degree(er_net), col = "#93c5fd", border = "white",
     main = "ER", xlab = "Degree")
hist(degree(ba_net), col = "#f9a8d4", border = "white",
     main = "BA", xlab = "Degree")

In [ ]:
# 3. 1000次模拟假设检验

real_g <- sample_smallworld(1, 38, 2, 0.05)
n <- vcount(real_g)
p <- edge_density(real_g)
real_cluster   <- transitivity(real_g, "global")
real_triangles <- sum(count_triangles(real_g)) / 3

sim_cluster   <- numeric(1000)
sim_triangles <- numeric(1000)
for (i in 1:1000) {
  g <- erdos.renyi.game(n, p, type = "gnp")
  sim_cluster[i]   <- transitivity(g, "global")
  sim_triangles[i] <- sum(count_triangles(g)) / 3
}

cat("聚集系数  —— 随机均值:", round(mean(sim_cluster, na.rm=T), 3),
    " 真实值:", round(real_cluster, 3),
    " p值:", mean(sim_cluster >= real_cluster, na.rm=T), "\n")
cat("三角形数量 —— 随机均值:", round(mean(sim_triangles), 1),
    " 真实值:", real_triangles,
    " p值:", mean(sim_triangles >= real_triangles), "\n")

par(mfrow = c(1, 2))
hist(sim_cluster, col = "#bfdbfe", border = "white",
     main = "聚集系数分布", xlab = "聚集系数")
abline(v = real_cluster, col = "red", lwd = 2, lty = 2)

hist(sim_triangles, col = "#bbf7d0", border = "white",
     main = "三角形数量分布", xlab = "三角形数量")
abline(v = real_triangles, col = "red", lwd = 2, lty = 2)

In [ ]:
# 3. 佛罗伦萨婚姻网络ERGM

data(florentine)
plot(flomarriage,
     vertex.cex = sqrt(flomarriage %v% "wealth") / 3,
     vertex.col = "#93c5fd", displaylabels = TRUE)

model1 <- ergm(flomarriage ~ edges)
summary(model1)

theta <- coef(model1)["edges"]


In [ ]:
# 4. 加入结构项与节点属性

model2 <- ergm(flomarriage ~ edges + gwesp(0.5, fixed = TRUE),
               control = control.ergm(seed = 42))

model3 <- ergm(flomarriage ~ edges + gwesp(0.5, fixed = TRUE) + nodecov("wealth"),
               control = control.ergm(seed = 42))

summary(model3)


In [ ]:
# 5. MCMC诊断与GOF

mcmc.diagnostics(model3, vars.per.page = 3)

set.seed(42)
gof3 <- gof(model3, control = control.gof.ergm(nsim = 100))
par(mfrow = c(2, 3))
plot(gof3)